In [1]:
# 02_preprocessing.ipynb
# Regulatory-Constrained Fairness Score for Credit Scoring
# Notebook 02 - Preprocessing with a THREE-WAY split.
#
# Reviewer-driven change: a train / validation / test split (60/20/20, stratified)
# replaces the previous two-way split. All post-hoc decision thresholds (baseline
# prevalence threshold and the reject-option group thresholds) are tuned on the
# VALIDATION split only and then frozen for the TEST split, so no fairness or
# performance number reported later is tuned on test data.
#
# Fitting discipline: median imputation, standard scaling and one-hot vocabularies
# are fit on the TRAIN split only, then applied unchanged to validation and test.
# Sensitive attributes are preserved as columns. The privileged reference group and
# the favourable outcome are recorded in the metadata consumed by later notebooks.
# No filesystem paths are printed (anonymity).

import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
VAL_SIZE = 0.20   # fraction of the full dataset
TEST_SIZE = 0.20  # fraction of the full dataset
FAVOURABLE_LABEL = 0

# Privileged reference group per sensitive attribute (fixed a priori, documented).
PRIVILEGED = {
    "sex": "male",
    "age_group": "old",
    "education": ["graduate", "university"],
}

df_german = pd.read_parquet(os.path.join(PROCESSED_DIR, "german.parquet"))
df_taiwan = pd.read_parquet(os.path.join(PROCESSED_DIR, "taiwan.parquet"))
df_home = pd.read_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval.parquet"))


def preprocess_dataset(df, target_col, sensitive_cols, drop_cols=None,
                       val_size=VAL_SIZE, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    # Three-way stratified split, then fit transforms on train only.
    df = df.copy()
    drop_cols = drop_cols or []
    y = df[target_col].astype(int)
    sensitive = df[sensitive_cols].copy()
    feature_df = df.drop(columns=[target_col] + sensitive_cols + drop_cols, errors="ignore")

    num_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()

    # First carve out the test split, then split the remainder into train and val.
    idx_tr_val, idx_test = train_test_split(
        feature_df.index, test_size=test_size, random_state=random_state, stratify=y)
    rel_val = val_size / (1.0 - test_size)
    idx_train, idx_val = train_test_split(
        idx_tr_val, test_size=rel_val, random_state=random_state, stratify=y.loc[idx_tr_val])

    def _slice(idx):
        return feature_df.loc[idx].copy()

    X_train_raw, X_val_raw, X_test_raw = _slice(idx_train), _slice(idx_val), _slice(idx_test)

    # Numeric: median imputation + standard scaling (fit on train).
    if num_cols:
        med = X_train_raw[num_cols].median()
        for part in (X_train_raw, X_val_raw, X_test_raw):
            part[num_cols] = part[num_cols].fillna(med)
        scaler = StandardScaler().fit(X_train_raw[num_cols])
        num_parts = [pd.DataFrame(scaler.transform(p[num_cols]), columns=num_cols, index=p.index)
                     for p in (X_train_raw, X_val_raw, X_test_raw)]
    else:
        num_parts = [pd.DataFrame(index=p.index) for p in (X_train_raw, X_val_raw, X_test_raw)]

    # Categorical: fill "missing", one-hot, align every split to the train vocabulary.
    if cat_cols:
        cats = []
        for p in (X_train_raw, X_val_raw, X_test_raw):
            d = pd.get_dummies(p[cat_cols].astype("object").fillna("missing"), drop_first=False)
            cats.append(d)
        train_cols = cats[0].columns
        cats = [c.reindex(columns=train_cols, fill_value=0) for c in cats]
    else:
        cats = [pd.DataFrame(index=p.index) for p in (X_train_raw, X_val_raw, X_test_raw)]

    parts = []
    for idx, num_p, cat_p in zip((idx_train, idx_val, idx_test), num_parts, cats):
        X = pd.concat([num_p, cat_p], axis=1)
        X[target_col] = y.loc[idx].values
        for c in sensitive_cols:
            X[c] = sensitive.loc[idx, c].values
        parts.append(X)
    train, val, test = parts

    meta = {
        "target_col": target_col, "sensitive_cols": sensitive_cols,
        "privileged": {c: PRIVILEGED[c] for c in sensitive_cols},
        "favourable_label": FAVOURABLE_LABEL,
        "n_features": train.shape[1] - 1 - len(sensitive_cols),
        "n_train": len(train), "n_val": len(val), "n_test": len(test),
        "num_cols": num_cols, "cat_cols": cat_cols,
    }
    return train, val, test, meta


specs = [
    ("german", df_german, "target", ["sex", "age_group"], ["personal_status_sex"]),
    ("taiwan", df_taiwan, "target", ["sex", "education"], ["SEX", "EDUCATION"]),
    ("credit_card_approval", df_home, "target", ["sex"],
     ["ID", "CODE_GENDER", "target_30dpd", "target_60dpd"]),
]

metadata = {"config": {"random_state": RANDOM_STATE, "val_size": VAL_SIZE,
                       "test_size": TEST_SIZE, "favourable_label": FAVOURABLE_LABEL}}
prep_rows = []
for key, df, tgt, sens, drop in specs:
    train, val, test, meta = preprocess_dataset(df, tgt, sens, drop)
    train.to_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet"))
    val.to_parquet(os.path.join(PROCESSED_DIR, f"{key}_val.parquet"))
    test.to_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet"))
    metadata[key] = meta
    prep_rows.append({"dataset": key, "n_features": meta["n_features"],
                      "n_train": meta["n_train"], "n_val": meta["n_val"],
                      "n_test": meta["n_test"], "sensitive": ", ".join(sens),
                      "privileged": "; ".join(f"{c}={meta['privileged'][c]}" for c in sens)})
    print(f"[{key}] features={meta['n_features']} "
          f"train/val/test={meta['n_train']}/{meta['n_val']}/{meta['n_test']}")

with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json"), "w") as f:
    json.dump(metadata, f, indent=2)

prep_summary = pd.DataFrame(prep_rows)
print(prep_summary.to_string(index=False))
prep_summary.to_csv(os.path.join(TAB_DIR, "02_preprocess_summary.csv"), index=False)
print("Notebook 02 complete.")


[german] features=57 train/val/test=600/200/200
[taiwan] features=21 train/val/test=18000/6000/6000


[credit_card_approval] features=53 train/val/test=21873/7292/7292
             dataset  n_features  n_train  n_val  n_test      sensitive                                     privileged
              german          57      600    200     200 sex, age_group                        sex=male; age_group=old
              taiwan          21    18000   6000    6000 sex, education sex=male; education=['graduate', 'university']
credit_card_approval          53    21873   7292    7292            sex                                       sex=male
Notebook 02 complete.
